[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/49_tokenizer_full_solution.ipynb)

# Solution: Tokenizer (Complete BPE Implementation)

Reference solution.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
# No imports needed

In [ ]:
# ✅ SOLUTION

class FullBPETokenizer:
    def __init__(self):
        self.special_tokens = ['<pad>', '<unk>', '<bos>', '<eos>']
        self.end_of_word = '</w>'
        self.merges = []

        self.token_to_id = {tok: i for i, tok in enumerate(self.special_tokens)}
        self.id_to_token = {i: tok for tok, i in self.token_to_id.items()}

    def _merge_once(self, symbols, pair):
        out = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == pair[0] and symbols[i + 1] == pair[1]:
                out.append(pair[0] + pair[1])
                i += 2
            else:
                out.append(symbols[i])
                i += 1
        return out

    def _apply_merges(self, symbols):
        for pair in self.merges:
            symbols = self._merge_once(symbols, pair)
        return symbols

    def train(self, corpus, num_merges):
        vocab = {}
        for text in corpus:
            for word in text.split():
                symbols = tuple(list(word) + [self.end_of_word])
                vocab[symbols] = vocab.get(symbols, 0) + 1

        self.merges = []
        for _ in range(num_merges):
            pair_counts = {}
            for word, freq in vocab.items():
                for i in range(len(word) - 1):
                    pair = (word[i], word[i + 1])
                    pair_counts[pair] = pair_counts.get(pair, 0) + freq

            if not pair_counts:
                break

            best_pair = max(pair_counts.items(), key=lambda kv: (kv[1], kv[0]))[0]
            self.merges.append(best_pair)

            new_vocab = {}
            for word, freq in vocab.items():
                merged = tuple(self._merge_once(list(word), best_pair))
                new_vocab[merged] = new_vocab.get(merged, 0) + freq
            vocab = new_vocab

        token_set = {self.end_of_word}
        for word in vocab:
            token_set.update(word)
        for a, b in self.merges:
            token_set.add(a + b)

        self.token_to_id = {tok: i for i, tok in enumerate(self.special_tokens)}
        for tok in sorted(token_set):
            if tok not in self.token_to_id:
                self.token_to_id[tok] = len(self.token_to_id)
        self.id_to_token = {i: tok for tok, i in self.token_to_id.items()}

    def encode(self, text, add_special_tokens=True):
        ids = []
        if add_special_tokens:
            ids.append(self.token_to_id['<bos>'])

        unk_id = self.token_to_id['<unk>']
        for word in text.split():
            symbols = list(word) + [self.end_of_word]
            symbols = self._apply_merges(symbols)
            for tok in symbols:
                ids.append(self.token_to_id.get(tok, unk_id))

        if add_special_tokens:
            ids.append(self.token_to_id['<eos>'])
        return ids

    def decode(self, token_ids, skip_special_tokens=True):
        tokens = []
        for idx in token_ids:
            tok = self.id_to_token.get(int(idx), '<unk>')
            if skip_special_tokens and tok in self.special_tokens:
                continue
            tokens.append(tok)

        words = []
        current = ''
        for tok in tokens:
            if tok in self.special_tokens:
                if current:
                    words.append(current)
                    current = ''
                words.append(tok)
                continue

            if tok == self.end_of_word:
                words.append(current)
                current = ''
            elif tok.endswith(self.end_of_word):
                current += tok[:-len(self.end_of_word)]
                words.append(current)
                current = ''
            else:
                current += tok

        if current:
            words.append(current)

        return ' '.join(w for w in words if w != '')


In [ ]:
# Demo
tok = FullBPETokenizer()
tok.train(['low', 'lower', 'lowest', 'newest'], num_merges=10)
ids = tok.encode('low lower')
print('Encoded ids:', ids)
print('Decoded:', tok.decode(ids))
print('Vocab size:', len(tok.token_to_id))

In [ ]:
from torch_judge import check
check('tokenizer_full')